### データセットからニュースの要約文と本文を取得する

In [3]:
!curl -O https://raw.githubusercontent.com/KodairaTomonori/ThreeLineSummaryDataset/master/data/train.csv
!move train.csv downloaded_train.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 3746k  100 3746k    0     0  6384k      0 --:--:-- --:--:-- --:--:-- 6460k


        1 個のファイルを移動しました。


In [ ]:
from urllib.request import urlopen
from bs4 import BeautifulSoup
import time

# 収集するニュース記事のインデックス
start_index = 0 # 開始インデックス
end_index = 15000 # 終了インデックス

# コンテンツの取得
def get_content(id):
    # サーバに負荷をかけないように10秒スリープ
    time.sleep(5)

    URL = 'https://news.livedoor.com/article/detail/' + id + '/'
    print(URL)
    try:
        with urlopen(URL) as res:
            html = res.read().decode('euc_jp', 'ignore')
            soup = BeautifulSoup(html, 'html.parser')

            # 「写真拡大」のリンクを削除
            for photo_expand in soup.select('.articleImageExpand'):
                photo_expand.decompose()

            # ページャー要素（1 2 次へ）を削除
            for pager in soup.select('.pager'):
                pager.decompose()

            # 画像リンクとそれに関連するdiv要素を削除
            for unwanted in soup.select('div.image-data-reference, p a[target="_blank"]'):
                unwanted.decompose()

            # 広告やスクリプトを削除
            for ad in soup.select('div.teads-adCall, div[id^="div-gpt-ad"], script'):
                ad.decompose()

            # 本文の抽出
            output1 = ''
            article_body = soup.select_one('.articleBody')
            if article_body:
                output1 = article_body.get_text(strip=True, separator=' ')
            else:
                return

            if not output1:  # 記事がない場合
                return
            output1 += '\n'

            # 要約の抽出
            output0 = ''
            summary_list = soup.select('.summaryList li')
            for summary in summary_list:
                output0 += summary.get_text(strip=True) + '\t'
            if not output0:  # 要約がない場合
                return

            # 出力
            print(output0 + output1)
            with open('output.tsv', mode='a', encoding='utf-8') as f:
                f.writelines(output0 + output1)

    except Exception as e:
        print(f'Exception: {e}')

# IDリストの生成の取得
id_list = []
with open('downloaded_train.csv', mode='r') as f:
    lines = f.readlines()
    for line in lines:
        id = line.strip().split(',')[3].split('.')[0]
        id_list.append(id)

# コンテンツの取得
for i in range(start_index, end_index):
    print('index:', i)
    get_content(id_list[i])

**memo**
- 昔(9年前)のニュース記事のため多くのニュースが既に削除されていた。